In [2]:
import cv2
import os
import random
import yaml

In [4]:
## Definindo os caminhhos
CAMINHO_IMAGENS = "copos_e_garrafas/train/images"
CAMINHO_LABELS = "copos_e_garrafas/train/labels"
ARQUIVO_YAML = "copos_e_garrafas/data.yaml"

## CLASSES
NOMES_CLASSES ={}

if os.path.exists(ARQUIVO_YAML):
    with open(ARQUIVO_YAML,'r', encoding='utf-8') as f:
        dados_yaml = yaml.safe_load(f)
        if 'names' in dados_yaml:
            if isinstance(dados_yaml['names'], list):
                NOMES_CLASSES={i: nome for i,nome in enumerate(dados_yaml['names'])}
            elif isinstance(dados_yaml['names'],dict):
                NOMES_CLASSES  = dados_yaml['names']
            print(f"Carregadas {len(NOMES_CLASSES)} classes do ficheiro YAML.")
            print(f" {NOMES_CLASSES}")
else:
    print(f'o arquivo em {CAMINHO_IMAGENS} não existe!')            


Carregadas 2 classes do ficheiro YAML.
 {0: 'copo', 1: 'garrafa'}


In [5]:
# 2. Função para gerar cores aleatórias vibrantes para cada classe
cores_classes = {}
def obter_cor(classe_id):
    if classe_id not in cores_classes:
        # Gera cores (B, G, R) mais claras para melhor contraste
        cores_classes[classe_id] = (random.randint(100, 255), random.randint(100, 255), random.randint(100, 255))
    return cores_classes[classe_id]

In [6]:
# 3. Listar todas as imagens na pasta
imagens = [img for img in os.listdir(CAMINHO_IMAGENS) if img.endswith(('.jpg', '.png', '.jpeg'))]

if not imagens:
    print(f"Nenhuma imagem encontrada em: {CAMINHO_IMAGENS}")
    exit()

print(f"Encontradas {len(imagens)} imagens. Pressiona qualquer tecla para avançar. Pressiona 'q' para sair.")

Encontradas 677 imagens. Pressiona qualquer tecla para avançar. Pressiona 'q' para sair.


In [8]:
# 4. Loop principal de renderização
for nome_img in imagens:
    caminho_img = os.path.join(CAMINHO_IMAGENS, nome_img)
    img = cv2.imread(caminho_img)
    
    if img is None:
        continue

    altura_img, largura_img = img.shape[:2]
    
    # Busca o .txt correspondente
    nome_txt = os.path.splitext(nome_img)[0] + ".txt"
    caminho_txt = os.path.join(CAMINHO_LABELS, nome_txt)

    # 5. Lê as anotações se o ficheiro existir
    if os.path.exists(caminho_txt):
        with open(caminho_txt, "r") as arquivo:
            linhas = arquivo.readlines()
            
            for linha in linhas:
                valores = linha.strip().split()
                if len(valores) == 5:
                    classe_id = int(valores[0])
                    x_centro = float(valores[1])
                    y_centro = float(valores[2])
                    largura = float(valores[3])
                    altura = float(valores[4])

                    # Converter de YOLO (normalizado) para Pixels
                    w = int(largura * largura_img)
                    h = int(altura * altura_img)
                    x_min = int((x_centro * largura_img) - (w / 2))
                    y_min = int((y_centro * altura_img) - (h / 2))
                    
                    # Desenhar a Bounding Box e a Label
                    cor = obter_cor(classe_id)
                    # Pega o nome no dicionário dinâmico do YAML ou usa o ID numérico se não existir
                    nome_classe = NOMES_CLASSES.get(classe_id, f"ID: {classe_id}")
                    
                    # Desenha o retângulo do objeto
                    cv2.rectangle(img, (x_min, y_min), (x_min + w, y_min + h), cor, 2)
                    
                    # Desenha um fundo preenchido para o texto (para facilitar a leitura)
                    tamanho_texto, _ = cv2.getTextSize(nome_classe, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
                    cv2.rectangle(img, (x_min, y_min - 25), (x_min + tamanho_texto[0], y_min), cor, -1)
                    
                    # Escreve o texto da classe
                    cv2.putText(img, nome_classe, (x_min, y_min - 7), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)

    # 6. Exibe a imagem no ecrã
    cv2.imshow("Visualizador de Dataset YOLO", img)
    
    # Espera utilizador pressionar uma tecla (0 = espera infinitamente)
    tecla = cv2.waitKey(0) & 0xFF
    
    # Se apertar 'q', quebra o loop e fecha o programa
    if tecla == ord('q'):
        print("A sair do visualizador...")
        break

cv2.destroyAllWindows()

A sair do visualizador...
